In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import chardet
import unicodedata
import json

# -------------------------
# Helper functions
# -------------------------
def clean_column(c):
    # Strip whitespace and normalize unicode
    return unicodedata.normalize('NFC', c.strip())

def get_columns(file_list, normalize=False, rename_map=None, file_encodings=None):
    columns_per_file = {}
    for f in file_list:
        encoding = file_encodings.get(f, 'utf-8')
        df = pd.read_csv(f, nrows=5, delimiter=";", encoding=encoding)
        cols = df.columns
        if normalize and rename_map:
            cols = [rename_map.get(clean_column(c), clean_column(c)) for c in cols]
        columns_per_file[f] = set(cols)
    return columns_per_file

# -------------------------
# Find all CSV files
# -------------------------
files = sorted([f for f in glob.glob("data/*onnettomu*.csv") if "osall" not in f])

# Detect encodings
file_encodings = {}
for file in files:
    with open(file, 'rb') as f:
        rawdata = f.read(10000)  # first 10 KB
        file_encodings[file] = chardet.detect(rawdata)['encoding']

# Optional: separate files
files_no2024 = [f for f in files if "2024" not in f]
files_2015_2021 = [f for f in files if any(str(y) in f for y in range(2015, 2022))]

# -------------------------
# Column rename mapping
# -------------------------
rename_map = {
    'Onnettomuuspaikka': 'Onnpaikka',
    'Osallisten lkm': 'Osallkm',
    'Liikennevalot': 'Liikvalot',
    'Näkemä% yli 150': 'Nakos150',
    'Näkemä% yli 300': 'Nakos300',
    'Näkemä% yli 450': 'Nakos460',
    'Maakunta ': 'Maakunta',
    'Toiminallinen luokka korjattu': 'Toiminnallinen luokka',
    'Onnettomuustyyppi nro': 'Onnettomuustyyppi_nro',
}

# -------------------------
# Common columns across all files (raw)
# -------------------------
raw_cols = get_columns(files, normalize=False, file_encodings=file_encodings)
common_raw = set.intersection(*raw_cols.values())
print("\nCommon columns (raw):", common_raw)
print("Amount:", len(common_raw))

# -------------------------
# Raw common columns excluding 2024
# -------------------------
raw_cols_no2024 = get_columns(files_no2024, normalize=False, file_encodings=file_encodings)
common_raw_no2024 = set.intersection(*raw_cols_no2024.values())
print("\nCommon columns (raw, no 2024):", common_raw_no2024)
print("Amount:", len(common_raw_no2024))

# -------------------------
# Normalized common columns excluding 2024
# -------------------------
norm_cols_no2024 = get_columns(files_no2024, normalize=True, rename_map=rename_map, file_encodings=file_encodings)
common_norm_no2024 = set.intersection(*norm_cols_no2024.values())
print("\nCommon columns (normalized, no 2024):", common_norm_no2024)
print("Amount:", len(common_norm_no2024))

# -------------------------
# Common columns for 2015-2021 (raw)
# -------------------------
cols_2015_2021 = get_columns(files_2015_2021, normalize=False, file_encodings=file_encodings)
common_2015_2021 = set.intersection(*cols_2015_2021.values())
print("\nCommon columns 2015-2021 (raw):", common_2015_2021)
print("Amount:", len(common_2015_2021))




Common columns (raw): set()
Amount: 0

Common columns (raw, no 2024): {'Vakavuus', 'Päällyste', 'Taajama', 'Sää', 'Pinta', 'Vuosi', 'Kunta', 'Rautatie', 'Tie', 'Tietyö', 'Valoisuus', 'Katuosoite'}
Amount: 12

Common columns (normalized, no 2024): {'Vakavuus', 'Päällyste', 'Taajama', 'Sää', 'Pinta', 'Vuosi', 'Kunta', 'Liikvalot', 'Rautatie', 'Tie', 'Maakunta', 'Onnpaikka', 'Osallkm', 'Tietyö', 'Valoisuus', 'Katuosoite'}
Amount: 16

Common columns 2015-2021 (raw): {'Talvhoitsel', 'Nakos460', 'Lisäkaisse', 'Onlksel', 'Y', 'Tienpitsel', 'Päällyslev', 'Toimlksel', 'Muuliit', 'Tietyyppi', 'Vuosi', 'Nopraj', 'Maakuntsel', 'Kvl', 'Mo_molsel', 'Liitlksel', 'Sääsel', 'Osallkm', 'Tietyö', 'Päällystlk', 'Nopsuunvas', 'Noplajisel', 'Rautatie', 'Kuntasel', 'Lähliittie', 'Elynimi', 'Solmutyyps', 'Valomsel', 'Maankäytse', 'Rautatsel', 'Valsel', 'Onluokka', 'Ontyyppi', 'Kk', 'Aosa', 'Kunta', 'Solmutyyp', 'Pintasel', 'Lämpötila', 'Päällsel', 'Risteys', 'Liikvalot', 'Nopsuunoik', 'Toimenpide', 'Onnpaiks

In [2]:
# -------------------------
# Parse 2015-2021 into combined DataFrame, use datapackage for dtypes for correct parsing
# -------------------------
# Load datapackage
with open("./data/datapackage.json", "r", encoding="utf-8") as f:
    dp = json.load(f)

# Build a dtype map from the JSON
dtype_map = {}
for resource in dp["resources"]:
    for field in resource["schema"]["fields"]:
        if field["type"] == "integer":
            dtype_map[field["name"]] = "Int64"   # pandas nullable int
        elif field["type"] == "number":
            dtype_map[field["name"]] = "float"
        else:
            dtype_map[field["name"]] = "string"

dfs = []
print("\nLoading 2015-2021 files with specified dtypes:")
for f in files_2015_2021:
    encoding = file_encodings.get(f, 'utf-8')
    # Only read columns that exist in the CSV and in the datapackage
    df = pd.read_csv(f, delimiter=';', encoding=encoding, low_memory=False, usecols=lambda c: c in dtype_map, dtype=dtype_map)
    
    # Ensure year column exists
    if 'Vuosi' not in df.columns:
        df['Vuosi'] = int(next((y for y in range(2015, 2022) if str(y) in f), None))
    print(f"Loaded {f}: shape {df.shape}, columns: {list(df.columns)}")
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f"\nCombined DataFrame shape 2015-2021: {combined_df.shape}")

## SEURAAVA POIS KUN LIITETÄÄN TAKAISIN prepare_data.ipynb:in TOKA SOLUUN!
combined_df.to_parquet("combined_2015_2021.parquet", index=False)






Loading 2015-2021 files with specified dtypes:
Loaded data\tieliikenneonnettomuudet_2015_onnettomuus.csv: shape (25175, 95), columns: ['Onnett_id', 'Tienpit', 'Tienpitsel', 'Tie', 'Aosa', 'Aet', 'Ajr', 'Vuosi', 'Kk', 'Vakavuusko', 'Vakavuus', 'ELY', 'Elynimi', 'Vkpv', 'Ontyyppi', 'Ontyypsel', 'Onluokka', 'Onlksel', 'Osallkm', 'Nopraj', 'Taajmerk', 'Taajamasel', 'Pinta', 'Pintasel', 'Valoisuus', 'Valsel', 'Sää', 'Sääsel', 'Onnpaikka', 'Onnpaiksel', 'Liikvalot', 'Liikvalsel', 'Liittyvtie', 'Maakunta', 'Maakuntsel', 'Kunta', 'Kuntasel', 'Noplaji', 'Noplajisel', 'Nopsuunvas', 'Nopsuunoik', 'Taajama', 'Mo_mol', 'Mo_molsel', 'Toimluokka', 'Toimlksel', 'Kvl', 'Raskaskvl', 'Tienlev', 'X', 'Y', 'Tietyö', 'Päällyste', 'Päällsel', 'Lämpötila', 'Risteys', 'Risteyssel', 'Rautatie', 'Rautatsel', 'Muuliit', 'Muuliitsel', 'Katuosoite', 'Tietyyppi', 'Tietyypsel', 'Talvhoitlk', 'Talvhoitsel', 'Tienverkas', 'Tienverkse', 'Maankäyttö', 'Maankäytse', 'Valoohjaus', 'Valoohjsel', 'Lisäkaisty', 'Lisäkaisse',

In [3]:
#